# Notebook 20 — Common support, paired controls, and budget-matched reruns

Standalone. Drop it anywhere in the repository and run top to bottom.

## What this notebook assumes about your files

It was built against an actual inventory of your repository, not an idealised one.
The important facts:

| Artifact | Status |
|---|---|
| `01_data/interim/splits/*.csv` | present — all six binary split files |
| `04_outputs/predictions/18_multiseed_cross_corpus/` | present — **90 item-level files, five seeds** (42, 1, 7, 123, 2024), both systems, all nine cells |
| Item-level predictions for seeds 2025/13/77/314/1337 | **absent** — only run-level summaries survive |
| Item-level predictions for LOCO, pooled-all, DANN | **absent** — only run-level summaries survive |
| Run-level per-seed tables (`19_*_runs.csv`, `18_*_runs.csv`) | present |
| All aggregate summary tables | present |
| `03_checkpoints/` | empty |

**Nothing below hard-fails on a missing file.** Each stage picks the best evidence tier
available and prints which one it used:

- **item-level** — re-scores actual per-item predictions
- **run-level** — uses per-seed scalars from the `*_runs.csv` tables, which is enough for
  any paired seed-level test
- **summary-level** — falls back to aggregate means and says so

A stage that genuinely cannot run prints the reason and continues rather than raising.

## Consequence for the manuscript

Section 5, Section 8 and the Data-availability statement currently promise item-level
predictions for the ten-seed matrix and the multi-source runs. **On this inventory that
promise is not true.** Stage T0-7 writes the corrected wording for you, derived from
what is actually on disk. Paste it into the manuscript before submitting.

## Cost

Stage T0 is CPU-only and takes about ten minutes.

Stage T1 needs a GPU. On a RunPod RTX 4090 at roughly \$0.40/hr, with \$4.80 available:

| Stage | Runs | Time | Cost |
|---|---|---|---|
| T1-1 budget-matched DANN | 15 | ~1 h | ~\$0.45 |
| T1-3 target-only *k*-shot control | 36 | ~40 min | ~\$0.30 |
| T1-2 XLM-R transfer matrix | 15 | ~1.5–2 h | ~\$0.80 |

All three together are under \$1.60, well inside budget. T1-4 from the previous draft has
been removed: it required source checkpoints, and `03_checkpoints/` is empty.

In [13]:
# ─────────────────────────────────────────────────────────────────────
#  Configuration and repository discovery
# ─────────────────────────────────────────────────────────────────────
import os, re, gc, json, math, time, random, warnings, unicodedata, hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score)

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)


def find_repo_root():
    """Locate the repository by looking for the frozen split files."""
    env = os.getenv("NB20_ROOT", "").strip()
    seen, candidates = set(), []
    for c in ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().resolve().parents]:
        if c not in seen:
            seen.add(c); candidates.append(c)
    for extra in ("/workspace/Sarcasm_detection", "/workspace", "/content",
                  "/kaggle/working", str(Path.home())):
        p = Path(extra)
        if p.exists() and p not in seen:
            seen.add(p); candidates.append(p)
    for c in candidates:
        if (c / "01_data" / "interim" / "splits").is_dir():
            return c.resolve()
    # last resort: search downward for the splits folder
    for c in candidates:
        try:
            for hit in c.glob("**/01_data/interim/splits"):
                return hit.parents[2].resolve()
        except Exception:
            continue
    raise RuntimeError(
        "Could not find the repository. Set NB20_ROOT to the folder that contains "
        "01_data/interim/splits/, e.g. os.environ['NB20_ROOT'] = '/workspace/Sarcasm_detection'")


ROOT   = find_repo_root()
SPLITS = ROOT / "01_data" / "interim" / "splits"
OUT    = ROOT / "04_outputs"
TABLES = OUT / "tables"
PRED   = OUT / "predictions"
FINAL  = OUT / "finalized_outputs"
FT, FF = FINAL / "tables", FINAL / "figures"
CKPT   = ROOT / "03_checkpoints" / "20_controls"
ARCH   = ROOT / "archives"
for p in (TABLES, PRED, FT, FF, CKPT):
    p.mkdir(parents=True, exist_ok=True)

CORPORA = ["ben_sarc_binary", "banglasarc_binary", "banglasarc3_binary"]
DISPLAY = {"ben_sarc_binary": "Ben-Sarc", "banglasarc_binary": "BanglaSarc",
           "banglasarc3_binary": "BanglaSarc3"}
MODEL_NAME = "csebuetnlp/banglabert"

SEEDS_10   = [42, 1, 7, 123, 2024, 2025, 13, 77, 314, 1337]
LOCO_SEEDS = [42, 1, 7, 123, 2024]

MAX_LENGTH, EPOCHS, PATIENCE = 128, 8, 2
BATCH_SIZE, EVAL_BATCH_SIZE = 32, 64
LEARNING_RATE, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.10
FEWSHOT_LR, FEWSHOT_BATCH = 2e-5, 16
K_GRID_CONTROL = [25, 50, 100, 250]

RUN = {
    # ── Stage T0: CPU only, ~10 minutes total ──
    "T0_0_inventory":        True,
    "T0_1_common_support":   True,
    "T0_2_paired_transfer":  True,
    "T0_3_collapse":         True,
    "T0_5_length_stats":     True,
    "T0_6_regen_check":      True,
    "T0_7_availability":     True,
    # ── Stage T1: GPU ──
    "T1_1_dann_matched":     True,   # ~1 h    ~$0.45   <- highest value
    "T1_3_target_only":      False,   # ~40 min ~$0.30
    "T1_2_xlmr_transfer":    True,   # ~1.5-2h ~$0.80
    # ── Figures ──
    "FIGURES":               True,
}

print("ROOT       :", ROOT)
for name, p in [("splits", SPLITS), ("predictions", PRED), ("tables", TABLES),
                ("finalized tables", FT), ("archives", ARCH)]:
    print(f"{name:<17}: {'OK ' if p.exists() else 'MISSING'} {p}")
print("\nstages enabled:", [k for k, v in RUN.items() if v])

ROOT       : /workspace/Cross-Corpus-Bengali-Sarcasm-Detection
splits           : OK  /workspace/Cross-Corpus-Bengali-Sarcasm-Detection/01_data/interim/splits
predictions      : OK  /workspace/Cross-Corpus-Bengali-Sarcasm-Detection/04_outputs/predictions
tables           : OK  /workspace/Cross-Corpus-Bengali-Sarcasm-Detection/04_outputs/tables
finalized tables : OK  /workspace/Cross-Corpus-Bengali-Sarcasm-Detection/04_outputs/finalized_outputs/tables
archives         : MISSING /workspace/Cross-Corpus-Bengali-Sarcasm-Detection/archives

stages enabled: ['T0_0_inventory', 'T0_1_common_support', 'T0_2_paired_transfer', 'T0_3_collapse', 'T0_5_length_stats', 'T0_6_regen_check', 'T0_7_availability', 'T1_1_dann_matched', 'T1_2_xlmr_transfer', 'FIGURES']


### Metric helpers

Behaviourally identical to Notebooks 18 and 19, so anything regenerated here matches the published tables.

In [14]:
_ZW = {ord(c): None for c in ["\u200b", "\u200c", "\u200d", "\ufeff"]}


def norm_key(value):
    if not isinstance(value, str):
        value = "" if pd.isna(value) else str(value)
    value = unicodedata.normalize("NFC", value).translate(_ZW)
    return re.sub(r"\s+", " ", value).strip().casefold()


def softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def score(gold, pred):
    cr = recall_score(gold, pred, labels=[0, 1], average=None, zero_division=0)
    return dict(accuracy=float(accuracy_score(gold, pred)),
                macro_f1=float(f1_score(gold, pred, average="macro", zero_division=0)),
                recall_class_0=float(cr[0]), recall_class_1=float(cr[1]))


def tci(values):
    v = np.asarray(values, dtype=float)
    n = len(v)
    if n == 0:
        return float("nan"), float("nan"), float("nan"), float("nan")
    m = float(v.mean()); sd = float(v.std(ddof=1)) if n > 1 else 0.0
    if n < 2:
        return m, sd, m, m
    h = stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)
    return m, sd, m - h, m + h


def expected_calibration_error(gold, probs, n_bins=15):
    gold = np.asarray(gold); probs = np.asarray(probs)
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    edges = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs((pred[mask] == gold[mask]).mean() - conf[mask].mean())
    return float(ece)


def exact_sign_flip_test(differences):
    d = np.asarray(differences, dtype=float)
    if len(d) == 0:
        return float("nan")
    if len(d) > 22:
        rng = np.random.default_rng(0)
        signs = rng.choice([-1.0, 1.0], size=(200000, len(d)))
        return float(np.mean(np.abs((d * signs).mean(axis=1)) >= abs(d.mean())))
    observed, vals = abs(d.mean()), []
    for bits in range(2 ** len(d)):
        s = np.array([1 if (bits >> i) & 1 else -1 for i in range(len(d))])
        vals.append(abs((d * s).mean()))
    return float(np.mean(np.asarray(vals) >= observed))


def holm(pvals):
    p = np.asarray(pvals, dtype=float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj


# ── splits ───────────────────────────────────────────────────────────
LABEL_CANDIDATES = ["label_binary", "label", "y", "target", "gold_label", "class"]
TEXT_CANDIDATES  = ["text", "comment", "sentence", "content", "Text"]


def read_split(corpus, split):
    path = SPLITS / f"{corpus}_{split}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    lab = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
    txt = next((c for c in TEXT_CANDIDATES if c in df.columns), None)
    if lab is None or txt is None:
        raise ValueError(f"{path.name}: need a text and a label column, found {list(df.columns)}")
    df = df.rename(columns={lab: "label_binary", txt: "text"})
    df["label_binary"] = df["label_binary"].astype(int)
    df["norm"] = df.text.map(norm_key)
    if "item_id" not in df.columns:
        df["item_id"] = [hashlib.sha1(f"{corpus}|{split}|{i}".encode()).hexdigest()[:16]
                         for i in range(len(df))]
    df["row_pos"] = np.arange(len(df))
    return df.reset_index(drop=True)


DATA = {}
for c in CORPORA:
    DATA[c] = {s: read_split(c, s) for s in ("train", "val", "test")}
    t = DATA[c]["test"]
    print(f"{DISPLAY[c]:<12} train {len(DATA[c]['train']):>6}  val {len(DATA[c]['val']):>5}  "
          f"test {len(t):>5}  test pos-rate {t.label_binary.mean():.3f}")


def clean_target_frame(sources, target):
    """Overlap-filtered target test set: drop items whose normalised key occurs
    anywhere in any source corpus. Diagonal cells are left untouched."""
    te = DATA[target]["test"].copy()
    if target in sources:
        return te.reset_index(drop=True), 0
    keys = set()
    for s in sources:
        for sp in ("train", "val", "test"):
            keys |= set(DATA[s][sp]["norm"])
    keep = ~te["norm"].isin(keys)
    return te[keep].reset_index(drop=True), int((~keep).sum())

Ben-Sarc     train  20498  val  2562  test  2563  test pos-rate 0.500
BanglaSarc   train   3708  val   463  test   464  test pos-rate 0.358
BanglaSarc3  train   6328  val   791  test   791  test pos-rate 0.499


## T0-0 — Artifact inventory

Establishes what evidence is available before anything tries to use it. Every later stage
consults `AVAIL` rather than guessing.

In [15]:
AVAIL = {}

if RUN["T0_0_inventory"]:
    # ── item-level prediction files, searched everywhere they might live ──
    PRED_GLOBS = [
        PRED / "18_multiseed_cross_corpus",
        PRED / "19_singlesource_newseeds",
        PRED / "19_loco_pooled",
        PRED / "19_dann",
        PRED / "20_dann_matched",
        ARCH / "notebook18_sanitized_review_package" / "predictions",
        ARCH / "text_predictions",
        PRED,
    ]
    FNAME_RE = re.compile(
        r"^(?:\d+[a-z]?_)?(?P<system>vanilla|fgm|dann|dannmatched|dann_matched)_"
        r"(?P<source>.+?)_seed(?P<seed>\d+)_to_(?P<target>.+?)\.csv$", re.I)

    found = {}
    for folder in PRED_GLOBS:
        if not folder.is_dir():
            continue
        for f in sorted(folder.glob("*.csv")):
            m = FNAME_RE.match(f.name)
            if not m:
                continue
            g = m.groupdict()
            key = (g["system"].lower().replace("_", ""), g["source"], g["target"], int(g["seed"]))
            found.setdefault(key, f)          # first location wins

    inv = pd.DataFrame([dict(system=k[0], source=k[1], target=k[2], seed=k[3],
                             path=str(v.relative_to(ROOT)))
                        for k, v in found.items()])
    AVAIL["item_level"] = inv
    if len(inv):
        print(f"Item-level prediction files discovered: {len(inv)}")
        print(f"  systems : {sorted(map(str, inv.system.unique()))}")
        print(f"  seeds   : {sorted(int(v) for v in inv.seed.unique())}")
        print(f"  cells   : {inv.groupby(['source','target']).ngroups} of 9")
        cov = inv.groupby("system").seed.nunique().to_dict()
        print(f"  seeds per system: {cov}")
    else:
        print("Item-level prediction files discovered: 0")

    # ── run-level per-seed tables ──
    RUN_TABLES = {
        "single_10seed":  ["19_singlesource_10seed_runs.csv"],
        "single_newseed": ["19_singlesource_newseed_runs.csv"],
        "single_5seed":   ["18_transformer_cross_corpus_multiseed_runs.csv"],
        "loco_pooled":    ["19_loco_pooled_runs.csv"],
        "dann":           ["19_dann_loco_runs.csv"],
        "label_eff":      ["19_label_efficiency_runs.csv"],
    }
    SUMMARY_TABLES = {
        "single_summary":   ["19_singlesource_10seed_summary.csv"],
        "loco_summary":     ["19_loco_pooled_summary.csv"],
        "pooled_vs_single": ["19_pooled_vs_single_source.csv"],
        "label_eff_summary":["19_label_efficiency_summary.csv"],
        "threshold":        ["19_threshold_recalibration.csv"],
        "neardup":          ["19_near_duplicate_sensitivity.csv"],
        "classical":        ["17_cross_corpus_classical_matrix.csv"],
        "llm":              ["18_llm_baselines.csv"],
        "shift_audit":      ["18_cross_corpus_shift_audit.csv"],
        "claim_ledger":     ["19_claim_ledger.csv"],
        "summary_5seed":    ["18_transformer_cross_corpus_multiseed_summary.csv"],
    }
    SEARCH_DIRS = [FT, TABLES, FINAL, OUT, ROOT, ARCH]


    def locate(names):
        for n in names:
            for d in SEARCH_DIRS:
                if not d.exists():
                    continue
                p = d / n
                if p.exists():
                    return p
            try:
                hits = sorted(ROOT.glob(f"**/{n}"))
                if hits:
                    return hits[0]
            except Exception:
                pass
        return None


    rows = []
    for label, names in {**RUN_TABLES, **SUMMARY_TABLES}.items():
        p = locate(names)
        AVAIL[label] = p
        if p is not None:
            try:
                n = len(pd.read_csv(p))
            except Exception:
                n = -1
            rows.append(dict(artifact=label, rows=n, path=str(p.relative_to(ROOT))))
        else:
            rows.append(dict(artifact=label, rows=None, path="MISSING"))
    print()
    print(pd.DataFrame(rows).to_string(index=False))

    AVAIL["item_seeds"] = sorted(int(v) for v in inv.seed.unique()) if len(inv) else []
    AVAIL["has_item"] = len(inv) > 0
    print(f"\nEvidence tier available for re-scoring: "
          f"{'ITEM-LEVEL (' + str(len(AVAIL['item_seeds'])) + ' seeds)' if AVAIL['has_item'] else 'RUN-LEVEL only'}")
else:
    print("skipped")

Item-level prediction files discovered: 135
  systems : ['dannmatched', 'fgm', 'vanilla']
  seeds   : [1, 7, 42, 123, 2024]
  cells   : 18 of 9
  seeds per system: {'dannmatched': 5, 'fgm': 5, 'vanilla': 5}

         artifact  rows                                                                                  path
    single_10seed 180.0                   04_outputs/finalized_outputs/tables/19_singlesource_10seed_runs.csv
   single_newseed  90.0                                    04_outputs/tables/19_singlesource_newseed_runs.csv
     single_5seed  90.0    04_outputs/finalized_outputs/tables/18_transformer_cross_corpus_multiseed_runs.csv
      loco_pooled 120.0                                             04_outputs/tables/19_loco_pooled_runs.csv
             dann  27.0                             04_outputs/finalized_outputs/tables/19_dann_loco_runs.csv
        label_eff 126.0                                        04_outputs/tables/19_label_efficiency_runs.csv
   single_summary  18.

### Tolerant loaders

Prediction files written by different notebooks use different column names. These
loaders normalise them, reconstruct missing metadata from the filename, and rebuild
`item_id` positionally against the overlap-filtered split when a file has none.

In [16]:
ID_CAND   = ["item_id", "id", "idx", "index", "row_id", "uid"]
GOLD_CAND = ["gold_label", "gold", "y_true", "true_label", "label_binary", "label", "target"]
PRED_CAND = ["pred_label", "pred", "y_pred", "prediction", "predicted_label", "pred_class"]
P1_CAND   = ["prob_1", "p1", "prob_sarcastic", "prob_pos", "prob_cal_1", "probability_1", "score"]


def _pick(cols, cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    return None


def load_one_prediction(path, system, source, target, seed):
    df = pd.read_csv(path)
    cid   = _pick(df.columns, ID_CAND)
    cgold = _pick(df.columns, GOLD_CAND)
    cpred = _pick(df.columns, PRED_CAND)
    cp1   = _pick(df.columns, P1_CAND)

    out = pd.DataFrame()
    if cgold is None:
        return None
    out["gold_label"] = pd.to_numeric(df[cgold], errors="coerce").astype("Int64")

    if cp1 is not None:
        out["prob_1"] = pd.to_numeric(df[cp1], errors="coerce")
    elif {"logit_0", "logit_1"}.issubset({c.lower() for c in df.columns}):
        l0 = _pick(df.columns, ["logit_0"]); l1 = _pick(df.columns, ["logit_1"])
        out["prob_1"] = softmax_np(df[[l0, l1]].to_numpy(dtype=float))[:, 1]
    else:
        out["prob_1"] = np.nan

    if cpred is not None:
        out["pred_label"] = pd.to_numeric(df[cpred], errors="coerce").astype("Int64")
    elif out["prob_1"].notna().any():
        out["pred_label"] = (out["prob_1"] >= 0.5).astype(int)
    else:
        return None

    if cid is not None:
        out["item_id"] = df[cid].astype(str)
    else:
        # Rebuild positionally against the overlap-filtered target split.
        frame, _ = clean_target_frame([source], target)
        if len(frame) == len(out):
            out["item_id"] = frame["item_id"].values
        else:
            out["item_id"] = [f"{target}#{i}" for i in range(len(out))]

    out["system"], out["source"] = system, source
    out["target"], out["seed"] = target, int(seed)
    out["pred_file"] = Path(path).name
    return out.dropna(subset=["gold_label", "pred_label"]).reset_index(drop=True)


def load_item_predictions(systems=None, seeds=None):
    """Concatenate every discovered item-level prediction file, optionally filtered."""
    inv = AVAIL.get("item_level")
    if inv is None or not len(inv):
        return None
    sel = inv.copy()
    if systems:
        sel = sel[sel.system.isin(systems)]
    if seeds:
        sel = sel[sel.seed.isin(seeds)]
    frames = []
    for _, r in sel.iterrows():
        try:
            f = load_one_prediction(ROOT / r.path, r.system, r.source, r.target, r.seed)
            if f is not None and len(f):
                frames.append(f)
        except Exception as e:
            print(f"  ! {Path(r.path).name}: {e}")
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)


RUNCOL_F1   = ["test_macro_f1", "macro_f1", "test_macro_F1"]
RUNCOL_SEED = ["seed", "random_seed"]
RUNCOL_N    = ["n_eval_clean", "n_eval", "n_test", "n_eval_total"]


def load_run_table(key):
    """Load a per-seed run table and normalise its column names."""
    p = AVAIL.get(key)
    if p is None:
        return None
    df = pd.read_csv(p)
    ren = {}
    for cands, std in ((RUNCOL_F1, "test_macro_f1"), (RUNCOL_SEED, "seed"),
                       (RUNCOL_N, "n_eval_clean")):
        c = _pick(df.columns, cands)
        if c and c != std:
            ren[c] = std
    return df.rename(columns=ren)


P_ITEM = load_item_predictions()
if P_ITEM is not None:
    print(f"Loaded {len(P_ITEM):,} item-level rows "
          f"across {P_ITEM.groupby(['system','source','target','seed']).ngroups} runs")
    print(f"  seeds: {sorted(int(v) for v in P_ITEM.seed.unique())}")
    print(f"  usable probabilities: {P_ITEM.prob_1.notna().mean():.1%} of rows")
else:
    print("No item-level predictions loaded; later stages will use run-level tables.")

Loaded 170,955 item-level rows across 135 runs
  seeds: [1, 7, 42, 123, 2024]
  usable probabilities: 100.0% of rows


## T0-1 — Common-support evaluation

The overlap filter removes a different number of items depending on the source, so two
source models scored on the same target are not scored on the same population:
BanglaSarc3 is evaluated at *n* = 791 from a Ben-Sarc source and *n* = 762 from a
BanglaSarc source. This intersects the surviving sets and re-scores every source model on
the common items.

Runs at whatever seed coverage the item-level files provide, and says which. The diagonal
is deliberately left alone: removing a corpus's own items from its own test set would make
the in-domain reference incomparable with the published literature.

In [17]:
common_support = None
if RUN["T0_1_common_support"]:
    if P_ITEM is None:
        print("No item-level predictions; common support cannot be computed.")
        print("It needs per-item identity, which run-level tables do not carry.")
        print("Reconstructing the populations analytically instead:")
        rows = []
        for target in CORPORA:
            sets = {}
            for s in CORPORA:
                frame, removed = clean_target_frame([s], target)
                sets[s] = set(frame.item_id)
            offs = [s for s in CORPORA if s != target]
            common = set.intersection(*[sets[s] for s in offs])
            for s in offs:
                rows.append(dict(source=s, target=target, n_own_filter=len(sets[s]),
                                 n_common_support=len(common),
                                 n_dropped=len(sets[s]) - len(common)))
        common_support = pd.DataFrame(rows)
        print(common_support.to_string(index=False))
        common_support.to_csv(FT / "20_common_support_populations.csv", index=False)
    else:
        rows = []
        for target in CORPORA:
            sub = P_ITEM[P_ITEM.target.eq(target)]
            offs = [s for s in CORPORA if s != target]
            per_source_sets = []
            for s in offs:
                ids = sub[sub.source.eq(s)].groupby("seed").item_id.apply(set)
                if len(ids):
                    per_source_sets.append(set.intersection(*ids.tolist()))
            if not per_source_sets:
                continue
            common = set.intersection(*per_source_sets)
            for system in sorted(sub.system.unique()):
                for s in offs:
                    cell = sub[sub.system.eq(system) & sub.source.eq(s)]
                    if cell.empty:
                        continue
                    own, com = [], []
                    for seed, g in cell.groupby("seed"):
                        own.append(score(g.gold_label.values, g.pred_label.values)["macro_f1"])
                        gg = g[g.item_id.isin(common)]
                        if len(gg):
                            com.append(score(gg.gold_label.values, gg.pred_label.values)["macro_f1"])
                    if not own or not com:
                        continue
                    mo, _, _, _ = tci(own)
                    mc, sdc, lo, hi = tci(com)
                    rows.append(dict(
                        system=system, source=s, target=target, seeds=len(own),
                        n_own_filter=int(cell.groupby("seed").item_id.nunique().max()),
                        n_common_support=len(common),
                        macro_f1_own_filter=mo, macro_f1_common_support=mc,
                        delta=mc - mo, common_std=sdc,
                        common_ci95_lo=lo, common_ci95_hi=hi))
        common_support = pd.DataFrame(rows)
        if len(common_support):
            common_support.to_csv(FT / "20_common_support_matrix.csv", index=False)
            print(f"Evidence tier: ITEM-LEVEL over seeds {sorted(int(v) for v in P_ITEM.seed.unique())}")
            print(f"Largest |change| from moving to common support: "
                  f"{common_support.delta.abs().max():.4f}")
            print(f"Cells changing by more than 0.005: "
                  f"{int((common_support.delta.abs() > 0.005).sum())} of {len(common_support)}\n")
            print(common_support[["system", "source", "target", "n_own_filter",
                                  "n_common_support", "macro_f1_own_filter",
                                  "macro_f1_common_support", "delta"]].to_string(index=False))
        else:
            print("No off-diagonal cells with item-level coverage.")
else:
    print("skipped")

Evidence tier: ITEM-LEVEL over seeds [1, 7, 42, 123, 2024]
Largest |change| from moving to common support: 0.0026
Cells changing by more than 0.005: 0 of 12

 system             source             target  n_own_filter  n_common_support  macro_f1_own_filter  macro_f1_common_support     delta
    fgm  banglasarc_binary    ben_sarc_binary          2563              2563             0.346016                 0.346016  0.000000
    fgm banglasarc3_binary    ben_sarc_binary          2563              2563             0.669969                 0.669969  0.000000
vanilla  banglasarc_binary    ben_sarc_binary          2563              2563             0.346176                 0.346176  0.000000
vanilla banglasarc3_binary    ben_sarc_binary          2563              2563             0.667526                 0.667526  0.000000
    fgm    ben_sarc_binary  banglasarc_binary           464               436             0.595584                 0.593029 -0.002555
    fgm banglasarc3_binary  banglasarc

## T0-2 — Paired negative transfer

The 15.8-point figure compares a ten-seed single-source mean against a five-seed pooled
mean. It does not have to.

**This stage does not need item-level predictions.** A paired seed-level test needs only
per-seed macro-F1 for both arms on the same evaluation population, and both
`19_singlesource_10seed_runs.csv` and `19_loco_pooled_runs.csv` carry exactly that.
The stage checks `n_eval_clean` agreement between arms and reports it, so you can see
whether the populations match rather than assuming they do.

It also reports **all six** source-addition contrasts rather than the single selected one,
which addresses the limitation that the effect rests on one comparison.

In [18]:
paired = None
if RUN["T0_2_paired_transfer"]:
    single = load_run_table("single_10seed")
    if single is None:
        single = load_run_table("single_5seed")
    loco = load_run_table("loco_pooled")

    if single is None or loco is None:
        print(f"Need both a single-source and a LOCO run table. "
              f"single={'ok' if single is not None else 'MISSING'}, "
              f"loco={'ok' if loco is not None else 'MISSING'}")
        if loco is None and AVAIL.get("loco_summary") is not None:
            ls = pd.read_csv(AVAIL["loco_summary"])
            print("\nFalling back to the LOCO summary (unpaired, means only):")
            print(ls.head(12).to_string(index=False))
    else:
        # Identify the pooled-source column and restrict to leave-one-corpus-out rows
        src_col = "source" if "source" in loco.columns else "source_corpora"
        if "protocol" in loco.columns:
            loco = loco[loco.protocol.astype(str).str.lower().str.contains("loco")]
        rows = []
        for held in CORPORA:
            pool = sorted([c for c in CORPORA if c != held])
            def _is_pool(v):
                return set(str(v).replace(",", "+").split("+")) == set(pool)
            for system in sorted(set(single.system.astype(str))):
                m = loco[loco[src_col].map(_is_pool) & loco.target.eq(held)]
                if "system" in loco.columns:
                    m = m[loco.system.astype(str).eq(system)] if len(m) else m
                if m.empty:
                    continue
                pooled_by_seed = m.groupby("seed").test_macro_f1.mean().to_dict()
                n_pool = int(m.n_eval_clean.max()) if "n_eval_clean" in m.columns else -1

                for base in pool:
                    o = single[single.system.astype(str).eq(system) &
                               single.source.eq(base) & single.target.eq(held)]
                    if o.empty:
                        continue
                    single_by_seed = o.groupby("seed").test_macro_f1.mean().to_dict()
                    n_single = int(o.n_eval_clean.max()) if "n_eval_clean" in o.columns else -1

                    shared = sorted(set(single_by_seed) & set(pooled_by_seed))
                    if len(shared) < 3:
                        continue
                    d = np.array([pooled_by_seed[s] - single_by_seed[s] for s in shared])
                    mm, sd, lo, hi = tci(d)
                    rows.append(dict(
                        system=system, held_out_target=held, base_source=base,
                        added_source=[c for c in pool if c != base][0],
                        n_paired_seeds=len(shared), seeds=",".join(map(str, shared)),
                        n_eval_single_arm=n_single, n_eval_pooled_arm=n_pool,
                        populations_match=bool(n_single == n_pool),
                        single_mean=float(np.mean([single_by_seed[s] for s in shared])),
                        pooled_mean=float(np.mean([pooled_by_seed[s] for s in shared])),
                        delta_pooled_minus_single=mm, delta_std=sd,
                        delta_ci95_lo=lo, delta_ci95_hi=hi,
                        exact_sign_flip_p=exact_sign_flip_test(d),
                        paired_t_p=float(stats.ttest_rel(
                            [pooled_by_seed[s] for s in shared],
                            [single_by_seed[s] for s in shared]).pvalue)
                        if len(shared) > 1 else float("nan")))

        paired = pd.DataFrame(rows)
        if len(paired):
            paired["p_holm"] = holm(paired.exact_sign_flip_p.fillna(1.0).values)
            paired = paired.sort_values("delta_pooled_minus_single").reset_index(drop=True)
            paired.to_csv(FT / "20_paired_negative_transfer.csv", index=False)
            print("Evidence tier: RUN-LEVEL (per-seed macro-F1), which is sufficient "
                  "for a paired seed test.\n")
            print(paired[["system", "held_out_target", "base_source", "added_source",
                          "n_paired_seeds", "populations_match", "single_mean",
                          "pooled_mean", "delta_pooled_minus_single",
                          "delta_ci95_lo", "delta_ci95_hi", "paired_t_p"]].to_string(index=False))
            w = paired.iloc[0]
            print(f"\nLargest negative transfer: adding "
                  f"{DISPLAY.get(w.added_source, w.added_source)} to "
                  f"{DISPLAY.get(w.base_source, w.base_source)} costs "
                  f"{-w.delta_pooled_minus_single:.4f} macro-F1 on held-out "
                  f"{DISPLAY.get(w.held_out_target, w.held_out_target)}, "
                  f"paired over {int(w.n_paired_seeds)} seeds, "
                  f"95% CI [{w.delta_ci95_lo:.4f}, {w.delta_ci95_hi:.4f}], "
                  f"paired t p = {w.paired_t_p:.4g}")
            if not paired.populations_match.all():
                print("\nNOTE: some contrasts have different n between arms because the "
                      "pooled filter removes items matching either source corpus. "
                      "Those rows are flagged populations_match = False.")
        else:
            print("No paired contrasts could be assembled; check the LOCO table's "
                  "source/protocol column values.")
            print("loco columns:", list(loco.columns))
else:
    print("skipped")

Evidence tier: RUN-LEVEL (per-seed macro-F1), which is sufficient for a paired seed test.

 system    held_out_target        base_source       added_source  n_paired_seeds  populations_match  single_mean  pooled_mean  delta_pooled_minus_single  delta_ci95_lo  delta_ci95_hi  paired_t_p
vanilla    ben_sarc_binary banglasarc3_binary  banglasarc_binary               5               True     0.667526     0.501785                  -0.165741      -0.185836      -0.145646    0.000022
    fgm    ben_sarc_binary banglasarc3_binary  banglasarc_binary               5               True     0.669969     0.508679                  -0.161289      -0.172626      -0.149952    0.000002
vanilla  banglasarc_binary banglasarc3_binary    ben_sarc_binary               5               True     0.641471     0.594217                  -0.047253      -0.108220       0.013714    0.097771
    fgm banglasarc3_binary    ben_sarc_binary  banglasarc_binary               5              False     0.649024     0.638649    

## T0-3 / T0-4 — Boundary placement or lost ranking?

Section 6.6 shows a threshold fitted on *source* validation data does not repair the
collapsed cells. That alone does not establish the scores carry no class information.

The **target-oracle threshold** uses target labels and is a diagnostic ceiling, not a
proposed method: if the boundary were placed perfectly, how much is recoverable? **AUROC**
and **AUPRC** answer the same question with no threshold at all.

Needs item-level probabilities, so it runs at whatever seed coverage exists and reports it.

In [19]:
collapse = None
if RUN["T0_3_collapse"]:
    if P_ITEM is None or P_ITEM.prob_1.notna().sum() == 0:
        print("Needs item-level probabilities and none were found.")
        print("Skipping without failing; the manuscript's cautious wording in 6.6 stands.")
    else:
        usable = P_ITEM[P_ITEM.prob_1.notna()]
        grid = np.linspace(0.01, 0.99, 197)
        rows = []
        for (system, src, tgt), cell in usable.groupby(["system", "source", "target"]):
            per_seed = []
            for seed, g in cell.groupby("seed"):
                gold = g.gold_label.values.astype(int)
                p1 = g.prob_1.values.astype(float)
                if len(np.unique(gold)) < 2:
                    continue
                arg = f1_score(gold, (p1 >= 0.5).astype(int), average="macro", zero_division=0)
                f1s = [f1_score(gold, (p1 >= t).astype(int), average="macro", zero_division=0)
                       for t in grid]
                bi = int(np.argmax(f1s))
                per_seed.append((arg, f1s[bi], grid[bi],
                                 float(roc_auc_score(gold, p1)),
                                 float(average_precision_score(gold, p1)),
                                 float((p1 >= 0.5).mean())))
            if not per_seed:
                continue
            a = np.array(per_seed, dtype=float)
            rows.append(dict(system=system, source=src, target=tgt,
                             in_domain=bool(src == tgt), seeds=len(a),
                             macro_f1_argmax=float(a[:, 0].mean()),
                             macro_f1_oracle_threshold=float(a[:, 1].mean()),
                             oracle_gain=float((a[:, 1] - a[:, 0]).mean()),
                             oracle_threshold_mean=float(a[:, 2].mean()),
                             auroc=float(a[:, 3].mean()), auprc=float(a[:, 4].mean()),
                             predicted_positive_rate=float(a[:, 5].mean())))
        collapse = pd.DataFrame(rows).sort_values(["system", "source", "target"])
        collapse.to_csv(FT / "20_collapse_diagnostics.csv", index=False)
        print(f"Evidence tier: ITEM-LEVEL over seeds {sorted(int(v) for v in usable.seed.unique())}\n")

        sysname = "fgm" if "fgm" in set(collapse.system) else collapse.system.iloc[0]
        off = collapse[(~collapse.in_domain) & collapse.system.eq(sysname)]
        print(off[["source", "target", "macro_f1_argmax", "macro_f1_oracle_threshold",
                   "oracle_gain", "auroc", "auprc",
                   "predicted_positive_rate"]].to_string(index=False))

        coll = off[off.macro_f1_argmax < 0.40]
        if len(coll):
            print("\nCollapsed cells:")
            for _, r in coll.iterrows():
                verdict = ("ranking SURVIVES - the operating point is wrong, so target-prior "
                           "estimation from unlabelled text is worth trying"
                           if r.auroc > 0.60 else
                           "ranking has ALSO collapsed - no threshold can recover this cell")
                print(f"  {DISPLAY.get(r.source, r.source)} -> {DISPLAY.get(r.target, r.target)}: "
                      f"argmax {r.macro_f1_argmax:.4f}, oracle {r.macro_f1_oracle_threshold:.4f}, "
                      f"AUROC {r.auroc:.4f}")
                print(f"      -> {verdict}")
        else:
            print("\nNo cell falls below 0.40 macro-F1 at this seed coverage.")
else:
    print("skipped")

Evidence tier: ITEM-LEVEL over seeds [1, 7, 42, 123, 2024]

            source             target  macro_f1_argmax  macro_f1_oracle_threshold  oracle_gain    auroc    auprc  predicted_positive_rate
banglasarc3_binary  banglasarc_binary         0.623804                   0.634244     0.010440 0.654893 0.486043                 0.460550
banglasarc3_binary    ben_sarc_binary         0.669886                   0.674816     0.004930 0.729793 0.704513                 0.567226
 banglasarc_binary banglasarc3_binary         0.334053                   0.362756     0.028703 0.531465 0.544749                 0.010761
 banglasarc_binary    ben_sarc_binary         0.346016                   0.372243     0.026227 0.469873 0.488629                 0.015529
   ben_sarc_binary banglasarc3_binary         0.649024                   0.668553     0.019529 0.714390 0.667720                 0.483944
   ben_sarc_binary  banglasarc_binary         0.595584                   0.645379     0.049795 0.683901 0.451110

## T0-5 — Class-conditional length

Reference [17] attributes structurally similar asymmetric collapse to class-conditional
length differences supporting shortcut learning. Section 6.3 stratifies *within-corpus*
accuracy by length, which tests the wrong thing: the hypothesis is about the
*between-corpus* difference in how length separates the classes.

Needs only the split files, so this always runs.

In [20]:
length_stats = None
if RUN["T0_5_length_stats"]:
    rows = []
    for c in CORPORA:
        for split in ("train", "test"):
            df = DATA[c][split].copy()
            df["n_char"] = df.text.astype(str).str.len()
            df["n_tok"] = df.text.astype(str).str.split().str.len()
            g0, g1 = df[df.label_binary.eq(0)], df[df.label_binary.eq(1)]
            if not len(g0) or not len(g1):
                continue
            sd0 = g0.n_char.std(ddof=1); sd1 = g1.n_char.std(ddof=1)
            pooled = math.sqrt((sd0 ** 2 + sd1 ** 2) / 2) if (sd0 > 0 or sd1 > 0) else 0.0
            rows.append(dict(
                corpus=c, split=split, n=len(df),
                char_mean_nonsarc=float(g0.n_char.mean()),
                char_mean_sarc=float(g1.n_char.mean()),
                tok_mean_nonsarc=float(g0.n_tok.mean()),
                tok_mean_sarc=float(g1.n_tok.mean()),
                char_ratio_sarc_over_nonsarc=float(g1.n_char.mean() / max(g0.n_char.mean(), 1e-9)),
                char_cohens_d=float((g1.n_char.mean() - g0.n_char.mean()) / pooled)
                if pooled > 0 else 0.0,
                mannwhitney_p=float(stats.mannwhitneyu(
                    g0.n_char.values, g1.n_char.values, alternative="two-sided").pvalue)))
    length_stats = pd.DataFrame(rows)
    length_stats.to_csv(FT / "20_class_conditional_length.csv", index=False)
    print(length_stats[["corpus", "split", "char_mean_nonsarc", "char_mean_sarc",
                        "char_ratio_sarc_over_nonsarc", "char_cohens_d",
                        "mannwhitney_p"]].to_string(index=False))

    tr = length_stats[length_stats.split.eq("train")]
    spread = float(tr.char_cohens_d.max() - tr.char_cohens_d.min())
    print(f"\nCohen's d across corpora (train): "
          f"{tr.char_cohens_d.min():+.3f} to {tr.char_cohens_d.max():+.3f}, spread {spread:.3f}")
    if spread > 0.30:
        print("The corpora disagree materially on how length separates the classes,")
        print("which is consistent with a length cue that does not transfer.")
    else:
        print("The corpora agree closely on this cue, so a length shortcut is unlikely")
        print("to explain the asymmetry; reference [17]'s mechanism is not supported here.")
    print("\nEither result is reportable. Write whichever the numbers give you.")
else:
    print("skipped")

            corpus split  char_mean_nonsarc  char_mean_sarc  char_ratio_sarc_over_nonsarc  char_cohens_d  mannwhitney_p
   ben_sarc_binary train          76.937165       79.432237                      1.032430       0.035775   5.139717e-21
   ben_sarc_binary  test          75.584699       78.387676                      1.037084       0.042480   4.961728e-04
 banglasarc_binary train          69.158824       70.490211                      1.019251       0.020261   2.571567e-52
 banglasarc_binary  test          80.550336       74.301205                      0.922420      -0.062534   1.014675e-10
banglasarc3_binary train          86.586305       59.892688                      0.691711      -0.469690   9.689971e-67
banglasarc3_binary  test          93.303030       56.359494                      0.604048      -0.483362   1.580362e-14

Cohen's d across corpora (train): -0.470 to +0.036, spread 0.505
The corpora disagree materially on how length separates the classes,
which is consistent with 

## T0-6 — Regeneration check

Contribution C5 claims every table is recomputable from released artifacts. This rebuilds
the transfer matrix from prediction files and diffs it against whichever published summary
covers the same seeds.

In [21]:
regen = None
if RUN["T0_6_regen_check"]:
    if P_ITEM is None:
        print("No item-level predictions, so regeneration cannot be demonstrated.")
        print("Contribution C5 must be scoped to what is actually released - see T0-7.")
    else:
        rows = []
        for (system, src, tgt), cell in P_ITEM.groupby(["system", "source", "target"]):
            vals, accs, r1s, eces = [], [], [], []
            for seed, g in cell.groupby("seed"):
                m = score(g.gold_label.values, g.pred_label.values)
                vals.append(m["macro_f1"]); accs.append(m["accuracy"])
                r1s.append(m["recall_class_1"])
                if g.prob_1.notna().all():
                    probs = np.column_stack([1 - g.prob_1.values, g.prob_1.values])
                    eces.append(expected_calibration_error(g.gold_label.values, probs))
            m_, sd, lo, hi = tci(vals)
            rows.append(dict(system=system, source=src, target=tgt,
                             in_domain=bool(src == tgt), seeds=len(vals),
                             macro_f1_mean=m_, macro_f1_std=sd,
                             macro_f1_ci95_lo=lo, macro_f1_ci95_hi=hi,
                             accuracy_mean=float(np.mean(accs)),
                             recall_class_1_mean=float(np.mean(r1s)),
                             ece_mean=float(np.mean(eces)) if eces else float("nan"),
                             n_eval_clean=int(cell.groupby("seed").item_id.nunique().max())))
        regen = pd.DataFrame(rows).sort_values(["system", "source", "target"])
        regen.to_csv(FT / "20_regenerated_summary.csv", index=False)
        n_seeds = int(regen.seeds.max())
        print(f"Regenerated {len(regen)} cells from predictions at {n_seeds} seeds.")

        # Compare against whichever published summary matches the seed coverage
        cand = "summary_5seed" if n_seeds <= 5 else "single_summary"
        pub_path = AVAIL.get(cand) or AVAIL.get("single_summary") or AVAIL.get("summary_5seed")
        if pub_path is not None:
            pub = pd.read_csv(pub_path)
            f1c = _pick(pub.columns, ["macro_f1_mean", "test_macro_f1", "macro_f1"])
            keys = [k for k in ["system", "source", "target"] if k in pub.columns]
            if f1c and len(keys) == 3:
                mg = regen.merge(pub[keys + [f1c]], on=keys, suffixes=("", "_pub"))
                mg["abs_diff"] = (mg.macro_f1_mean - mg[f1c]).abs()
                worst = float(mg.abs_diff.max()) if len(mg) else float("nan")
                print(f"Compared against {Path(pub_path).name} "
                      f"({pub.get('seeds', pd.Series([np.nan])).max()} seeds)")
                print(f"Largest disagreement: {worst:.2e}")
                if worst < 1e-6:
                    print("MATCH - tables regenerate exactly from predictions.")
                elif worst < 5e-3:
                    print("CLOSE - differences consistent with a different seed subset, "
                          "not a computation error.")
                else:
                    print("MISMATCH - investigate before submitting.")
                    print(mg.nlargest(5, "abs_diff")[keys + ["macro_f1_mean", f1c,
                                                            "abs_diff"]].to_string(index=False))
            else:
                print(f"Could not align columns with {Path(pub_path).name}.")
        else:
            print("No published summary found to compare against.")
else:
    print("skipped")

Regenerated 27 cells from predictions at 5 seeds.
Compared against 18_transformer_cross_corpus_multiseed_summary.csv (5 seeds)
Largest disagreement: 0.00e+00
MATCH - tables regenerate exactly from predictions.


## T0-7 — Corrected data-availability wording

The manuscript currently promises item-level predictions for the ten-seed matrix, the
multi-source runs and the domain-adversarial runs. This stage checks that against the
filesystem and writes wording that matches reality.

**Read the output and paste it into Section 5, Section 8 and the Data-availability
statement.** Promising artifacts that are not in the repository is the one reviewer-facing
error that is entirely self-inflicted.

In [22]:
if RUN["T0_7_availability"]:
    inv = AVAIL.get("item_level")
    have = {}
    if inv is not None and len(inv):
        for system, g in inv.groupby("system"):
            have[system] = dict(seeds=sorted(int(v) for v in g.seed.unique()),
                                cells=g.groupby(["source", "target"]).ngroups,
                                files=len(g))
    print("=" * 74)
    print("ITEM-LEVEL PREDICTION COVERAGE ON DISK")
    print("=" * 74)
    if have:
        for s, d in have.items():
            print(f"  {s:<12} {d['files']:>4} files | {d['cells']} cells | seeds {d['seeds']}")
    else:
        print("  none")
    for label, desc in [("loco_pooled", "leave-one-corpus-out / pooled"),
                        ("dann", "domain-adversarial"),
                        ("label_eff", "label-efficiency")]:
        p = AVAIL.get(label)
        print(f"  {desc:<28} item-level: NO | run-level table: "
              f"{'yes (' + Path(p).name + ')' if p else 'no'}")

    seeds = sorted({s for d in have.values() for s in d["seeds"]})
    n_files = sum(d["files"] for d in have.values())

    print()
    print("=" * 74)
    print("PASTE THIS INTO THE MANUSCRIPT (Section 5, retention paragraph)")
    print("=" * 74)
    txt = (
        f"Item-level predictions were retained for the {len(seeds)}-seed cross-corpus "
        f"matrix, comprising {n_files} prediction files covering both systems and all "
        f"nine directed cells. Each carries a stable row identifier, the gold label, the "
        f"predicted label and the predicted probability. For the remaining five seeds of "
        f"the ten-seed matrix, and for the multi-source, domain-adversarial and "
        f"label-efficiency runs, per-seed result tables were retained with per-file "
        f"checksums but item-level predictions were not. Paired seed-level tests remain "
        f"computable from the per-seed tables; item-level re-scoring, such as the "
        f"near-duplicate analysis of Section 6.8, is reproducible only over the "
        f"{len(seeds)} seeds for which predictions survive."
    )
    print(txt)
    print()
    print("=" * 74)
    print("AND THIS INTO THE DATA-AVAILABILITY STATEMENT")
    print("=" * 74)
    txt2 = (
        f"Item-level predictions are available for the {len(seeds)}-seed cross-corpus "
        f"matrix ({n_files} files). For the remaining seeds and for the multi-source, "
        f"domain-adversarial and label-efficiency runs, per-seed and aggregate summary "
        f"tables are available with checksums and item-level predictions were not retained."
    )
    print(txt2)
    (FT / "20_data_availability_wording.txt").write_text(
        txt + "\n\n" + txt2 + "\n", encoding="utf-8")
    print(f"\nAlso written to {(FT / '20_data_availability_wording.txt').relative_to(ROOT)}")

    # claim-ledger status consistency
    cl = AVAIL.get("claim_ledger")
    if cl is not None:
        led = pd.read_csv(cl)
        if "status" in led.columns and led.status.astype(str).str.lower().eq("confirmatory").any():
            n = int(led.status.astype(str).str.lower().eq("confirmatory").sum())
            print(f"\nWARNING: {Path(cl).name} marks {n} rows status='confirmatory', but the "
                  f"manuscript describes itself as a retrospective benchmark.")
            print("Change that column to 'final' so the artifact and the text agree.")
else:
    print("skipped")

ITEM-LEVEL PREDICTION COVERAGE ON DISK
  dannmatched    45 files | 9 cells | seeds [1, 7, 42, 123, 2024]
  fgm            45 files | 9 cells | seeds [1, 7, 42, 123, 2024]
  vanilla        45 files | 9 cells | seeds [1, 7, 42, 123, 2024]
  leave-one-corpus-out / pooled item-level: NO | run-level table: yes (19_loco_pooled_runs.csv)
  domain-adversarial           item-level: NO | run-level table: yes (19_dann_loco_runs.csv)
  label-efficiency             item-level: NO | run-level table: yes (19_label_efficiency_runs.csv)

PASTE THIS INTO THE MANUSCRIPT (Section 5, retention paragraph)
Item-level predictions were retained for the 5-seed cross-corpus matrix, comprising 135 prediction files covering both systems and all nine directed cells. Each carries a stable row identifier, the gold label, the predicted label and the predicted probability. For the remaining five seeds of the ten-seed matrix, and for the multi-source, domain-adversarial and label-efficiency runs, per-seed result tables 

---

# Stage T1 — GPU

Everything below trains models. On a RunPod RTX 4090 at about \$0.40/hr the three stages
total roughly \$1.55, well inside \$4.80.

Enable them in the configuration cell at the top. If no GPU is present the setup cell says
so and the training stages skip cleanly rather than crashing.

In [23]:
GPU_READY = True
if any(RUN[k] for k in ("T1_1_dann_matched", "T1_3_target_only", "T1_2_xlmr_transfer")):
    try:
        import torch
        import torch.nn as nn
        from transformers import (AutoTokenizer, AutoModel,
                                  AutoModelForSequenceClassification,
                                  TrainingArguments, Trainer, EarlyStoppingCallback)
        HAS_CUDA = torch.cuda.is_available()
        HAS_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
        DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
        print("torch", torch.__version__, "| device:", DEVICE, "| bf16:", HAS_BF16)
        if HAS_CUDA:
            print("gpu:", torch.cuda.get_device_name(0),
                  f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
        else:
            print("No CUDA device. Training stages would take many hours on CPU;")
            print("leave them disabled unless you know that is what you want.")

        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

        def set_seed(s):
            random.seed(s); np.random.seed(s); torch.manual_seed(s)
            if HAS_CUDA:
                torch.cuda.manual_seed_all(s)

        class EncodedDataset(torch.utils.data.Dataset):
            def __init__(self, frame, tok, max_len=MAX_LENGTH,
                         with_domain=False, domain_map=None):
                self.enc = tok(list(frame.text.astype(str)), truncation=True,
                               max_length=max_len, padding="max_length")
                self.labels = frame.label_binary.values.astype(int)
                self.domain = (frame.corpus.map(domain_map).values.astype(int)
                               if with_domain else None)

            def __len__(self):
                return len(self.labels)

            def __getitem__(self, i):
                item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
                item["labels"] = torch.tensor(self.labels[i])
                if self.domain is not None:
                    item["domain"] = torch.tensor(self.domain[i])
                return item

        def source_frames(sources, seed=42):
            tr = pd.concat([DATA[c]["train"].assign(corpus=c) for c in sources],
                           ignore_index=True)
            va = pd.concat([DATA[c]["val"].assign(corpus=c) for c in sources],
                           ignore_index=True)
            return (tr.sample(frac=1.0, random_state=seed).reset_index(drop=True),
                    va.reset_index(drop=True))

        def fit_temperature(logits, gold):
            logits = np.asarray(logits, dtype=np.float64); gold = np.asarray(gold, dtype=int)

            def obj(log_t):
                p = softmax_np(logits / np.exp(log_t))
                return float(-np.log(np.clip(p[np.arange(len(gold)), gold], 1e-12, 1.0)).mean())

            return float(np.exp(minimize_scalar(obj, bounds=(-2.3, 2.3), method="bounded").x))

        def free():
            gc.collect()
            if HAS_CUDA:
                torch.cuda.empty_cache()

        GPU_READY = True
        print("GPU helpers ready")
    except ImportError as e:
        print("torch/transformers unavailable:", e)
        print("Install them, or leave the T1 stages disabled and use Stage T0 only.")
else:
    print("All T1 stages disabled; skipping GPU setup.")

torch 2.8.0+cu128 | device: cuda | bf16: True
gpu: NVIDIA GeForce RTX 4090 | 25.3 GB


GPU helpers ready


## T1-1 — Budget-matched domain-adversarial training

**This is a correction, not an addition.** Notebook 19 called `run_dann(..., epochs=4)`
with no early stopping, while pooled leave-one-corpus-out training used `EPOCHS = 8` with
`patience = 2` and best-checkpoint selection on source validation macro-F1. The
domain-adversarial arm received half the optimisation budget, so its reported 0.022
shortfall cannot be cleanly attributed to the adversarial objective.

Everything here is byte-for-byte Notebook 19's `run_dann` except the budget: eight epochs,
early stopping with patience 2, best checkpoint by source validation macro-F1, on the same
five seeds as pooled training.

15 trainings, roughly one hour, about \$0.45.

In [24]:
dann_matched = None
if RUN["T1_1_dann_matched"] and GPU_READY:
    class GradReverse(torch.autograd.Function):
        @staticmethod
        def forward(ctx, x, lambd):
            ctx.lambd = lambd
            return x.view_as(x)

        @staticmethod
        def backward(ctx, grad_output):
            return grad_output.neg() * ctx.lambd, None

    class DANNModel(nn.Module):
        def __init__(self, model_name, n_labels=2, n_domains=2, dropout=0.1):
            super().__init__()
            self.encoder = AutoModel.from_pretrained(model_name)
            hidden = self.encoder.config.hidden_size
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Linear(hidden, n_labels)
            self.domain_head = nn.Sequential(
                nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(256, n_domains))

        def forward(self, input_ids, attention_mask, token_type_ids=None, lambd=0.0):
            kw = dict(input_ids=input_ids, attention_mask=attention_mask)
            if token_type_ids is not None:
                kw["token_type_ids"] = token_type_ids
            h = self.encoder(**kw).last_hidden_state[:, 0]
            h = self.dropout(h)
            return self.classifier(h), self.domain_head(GradReverse.apply(h, lambd))

    @torch.no_grad()
    def dann_logits(model, frame, batch=EVAL_BATCH_SIZE):
        model.eval()
        dl = torch.utils.data.DataLoader(EncodedDataset(frame, tokenizer),
                                         batch_size=batch, shuffle=False)
        outs = []
        for b in dl:
            b.pop("labels")
            b = {k: v.to(DEVICE) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                enabled=bool(HAS_CUDA)):
                lg, _ = model(**b)
            outs.append(lg.float().cpu().numpy())
        return np.concatenate(outs, axis=0)

    def run_dann_matched(sources, held, seed, epochs=EPOCHS, patience=PATIENCE,
                         lr=LEARNING_RATE, batch=BATCH_SIZE, dann_alpha=1.0):
        set_seed(seed)
        dmap = {c: i for i, c in enumerate(sources)}
        tr, va = source_frames(list(sources), seed=seed)
        dl = torch.utils.data.DataLoader(
            EncodedDataset(tr, tokenizer, with_domain=True, domain_map=dmap),
            batch_size=batch, shuffle=True, drop_last=True)
        model = DANNModel(MODEL_NAME, n_domains=len(sources)).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
        total = max(1, epochs * len(dl))
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=lr, total_steps=total, pct_start=WARMUP_RATIO,
            anneal_strategy="linear")
        ce = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler("cuda", enabled=bool(HAS_CUDA and not HAS_BF16))

        best_f1, best_state, bad, best_epoch, step = -1.0, None, 0, 0, 0
        t0 = time.time()
        for ep in range(epochs):
            model.train()
            for b in dl:
                p = step / total
                lambd = dann_alpha * (2.0 / (1.0 + math.exp(-10 * p)) - 1.0)
                labels = b.pop("labels").to(DEVICE); domain = b.pop("domain").to(DEVICE)
                b = {k: v.to(DEVICE) for k, v in b.items()}
                opt.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                    enabled=bool(HAS_CUDA)):
                    lg, dlg = model(**b, lambd=lambd)
                    loss = ce(lg, labels) + ce(dlg, domain)
                if scaler.is_enabled():
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
                sched.step(); step += 1

            vf1 = f1_score(va.label_binary.values,
                           dann_logits(model, va).argmax(axis=1),
                           average="macro", zero_division=0)
            print(f"    epoch {ep+1}/{epochs}  val macro-F1 {vf1:.4f}")
            if vf1 > best_f1 + 1e-6:
                best_f1, best_epoch, bad = vf1, ep + 1, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    print(f"    early stop (best epoch {best_epoch}, {best_f1:.4f})")
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        seconds = float(time.time() - t0)

        temp = fit_temperature(dann_logits(model, va), va.label_binary.values)
        outdir = PRED / "20_dann_matched"; outdir.mkdir(parents=True, exist_ok=True)
        rows = []
        for target in CORPORA:
            clean, removed = clean_target_frame(list(sources), target)
            lg = dann_logits(model, clean)
            pred = lg.argmax(axis=1)
            m = score(clean.label_binary.values, pred)
            probs = softmax_np(lg)
            fname = f"20_dannmatched_{'+'.join(sources)}_seed{seed}_to_{target}.csv"
            pd.DataFrame({
                "item_id": clean.item_id.values, "gold_label": clean.label_binary.values,
                "pred_label": pred, "logit_0": lg[:, 0], "logit_1": lg[:, 1],
                "prob_0": probs[:, 0], "prob_1": probs[:, 1], "temperature": temp,
                "system": "dann_matched", "source": "+".join(sources),
                "target": target, "seed": seed}).to_csv(outdir / fname, index=False)
            rows.append(dict(system="dann_matched", protocol="loco",
                             source="+".join(sources), held_out_corpus=held,
                             target=target, seed=int(seed),
                             held_out=bool(target not in sources),
                             n_removed_all_source=removed, n_eval_clean=len(clean),
                             epochs_budget=epochs, best_epoch=best_epoch,
                             val_macro_f1=float(best_f1),
                             test_macro_f1=m["macro_f1"], test_accuracy=m["accuracy"],
                             test_recall_class_1=m["recall_class_1"], temperature=temp,
                             ece=expected_calibration_error(clean.label_binary.values, probs),
                             train_seconds=seconds,
                             prediction_path=str((outdir / fname).relative_to(ROOT))))
        del model, best_state
        free()
        return rows

    all_rows, t_start = [], time.time()
    for held in CORPORA:
        srcs = sorted([c for c in CORPORA if c != held])
        for seed in LOCO_SEEDS:
            print(f"[DANN matched] held out {DISPLAY[held]} | seed {seed}")
            all_rows += run_dann_matched(srcs, held, seed)
    print(f"\ntotal wall time {(time.time()-t_start)/60:.1f} min")

    dann_matched = pd.DataFrame(all_rows)
    dann_matched.to_csv(TABLES / "20_dann_matched_runs.csv", index=False)

    summ = (dann_matched[dann_matched.held_out].groupby("held_out_corpus")
            .test_macro_f1.agg(["mean", "std", "count"]).reset_index())
    print("\nBudget-matched DANN on held-out targets:")
    print(summ.to_string(index=False))

    loco_s = AVAIL.get("loco_summary")
    if loco_s is not None:
        pooled = pd.read_csv(loco_s)
        for col, val in (("system", "fgm"), ("protocol", "loco")):
            if col in pooled.columns:
                pooled = pooled[pooled[col].astype(str).eq(val)]
        if "held_out" in pooled.columns:
            pooled = pooled[pooled.held_out.astype(bool)]
        f1c = _pick(pooled.columns, ["macro_f1_mean", "test_macro_f1"])
        cmp_ = summ.merge(pooled[["target", f1c]], left_on="held_out_corpus",
                          right_on="target", how="left")
        cmp_["delta_vs_pooled"] = cmp_["mean"] - cmp_[f1c]
        cmp_ = cmp_.rename(columns={f1c: "macro_f1_mean"})
        cmp_.to_csv(FT / "20_dann_matched_vs_pooled.csv", index=False)
        print("\nAgainst pooled LOCO at the same budget:")
        print(cmp_[["held_out_corpus", "mean", "macro_f1_mean",
                    "delta_vs_pooled"]].to_string(index=False))
        print(f"\nMean delta {cmp_.delta_vs_pooled.mean():+.4f} "
              f"(Notebook 19 reported -0.0216 at half the epoch budget)")
        if cmp_.delta_vs_pooled.mean() < -0.005:
            print("-> The ordering holds at matched budget. Report it as a clean finding.")
        else:
            print("-> The ordering does NOT hold at matched budget. The published 0.022")
            print("   shortfall was a budget artefact; correct Sections 6.6 and 9.")
elif RUN["T1_1_dann_matched"]:
    print("GPU not ready; skipped.")
else:
    print("skipped")

[DANN matched] held out Ben-Sarc | seed 42


    epoch 1/8  val macro-F1 0.7804
    epoch 2/8  val macro-F1 0.8271
    epoch 3/8  val macro-F1 0.7676
    epoch 4/8  val macro-F1 0.8029
    early stop (best epoch 2, 0.8271)
[DANN matched] held out Ben-Sarc | seed 1
    epoch 1/8  val macro-F1 0.7723
    epoch 2/8  val macro-F1 0.8110
    epoch 3/8  val macro-F1 0.7998
    epoch 4/8  val macro-F1 0.8203
    epoch 5/8  val macro-F1 0.8130
    epoch 6/8  val macro-F1 0.8159
    early stop (best epoch 4, 0.8203)
[DANN matched] held out Ben-Sarc | seed 7
    epoch 1/8  val macro-F1 0.8083
    epoch 2/8  val macro-F1 0.8043
    epoch 3/8  val macro-F1 0.8143
    epoch 4/8  val macro-F1 0.8035
    epoch 5/8  val macro-F1 0.7951
    early stop (best epoch 3, 0.8143)
[DANN matched] held out Ben-Sarc | seed 123
    epoch 1/8  val macro-F1 0.8055
    epoch 2/8  val macro-F1 0.8168
    epoch 3/8  val macro-F1 0.8258
    epoch 4/8  val macro-F1 0.7945
    epoch 5/8  val macro-F1 0.7615
    early stop (best epoch 3, 0.8258)
[DANN matched] held 

## T1-3 — Target-only *k*-shot control

The label-efficiency curves fine-tune a **source-trained** model on *k* target examples.
Without a model trained on the same examples from the **pretrained encoder only**, the
curves cannot separate two explanations: source training helped, or *k* target labels are
enough on their own.

This is the control that licenses the word "repair", which is why the title currently says
"Adaptation". Run it and the stronger word becomes available, or the weaker one becomes
provably correct.

36 small trainings, roughly 40 minutes, about \$0.30.

In [25]:
target_only = None
if RUN["T1_3_target_only"] and GPU_READY:
    def draw_k(target, k, seed):
        tr = DATA[target]["train"]
        rng = np.random.default_rng(10_000 + seed)
        per = max(1, k // 2)
        parts = []
        for lab in (0, 1):
            pool = tr[tr.label_binary.eq(lab)]
            take = min(per, len(pool))
            parts.append(pool.iloc[rng.choice(len(pool), size=take, replace=False)])
        return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True).head(k)

    def train_target_only(target, k, seed):
        set_seed(seed)
        sub = draw_k(target, k, seed)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2).to(DEVICE)
        dl = torch.utils.data.DataLoader(
            EncodedDataset(sub, tokenizer),
            batch_size=min(FEWSHOT_BATCH, max(2, len(sub))), shuffle=True)
        opt = torch.optim.AdamW(model.parameters(), lr=FEWSHOT_LR, weight_decay=WEIGHT_DECAY)
        ce = nn.CrossEntropyLoss()
        epochs = 8 if k <= 100 else (5 if k <= 500 else 3)
        t0 = time.time(); model.train()
        for _ in range(epochs):
            for b in dl:
                labels = b.pop("labels").to(DEVICE)
                b = {kk: v.to(DEVICE) for kk, v in b.items()}
                opt.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                    enabled=bool(HAS_CUDA)):
                    loss = ce(model(**b).logits, labels)
                loss.backward(); opt.step()
        seconds = float(time.time() - t0)

        te = DATA[target]["test"]
        dl2 = torch.utils.data.DataLoader(EncodedDataset(te, tokenizer),
                                          batch_size=EVAL_BATCH_SIZE, shuffle=False)
        model.eval(); outs = []
        with torch.no_grad():
            for b in dl2:
                b.pop("labels")
                b = {kk: v.to(DEVICE) for kk, v in b.items()}
                with torch.autocast("cuda", dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                    enabled=bool(HAS_CUDA)):
                    outs.append(model(**b).logits.float().cpu().numpy())
        m = score(te.label_binary.values, np.concatenate(outs, axis=0).argmax(axis=1))
        del model
        free()
        return dict(system="target_only", target=target, k=int(k), seed=int(seed),
                    n_drawn=len(sub), epochs=epochs, test_macro_f1=m["macro_f1"],
                    test_accuracy=m["accuracy"], test_recall_class_1=m["recall_class_1"],
                    train_seconds=seconds)

    rows, t_start = [], time.time()
    for target in CORPORA:
        for k in K_GRID_CONTROL:
            for seed in [42, 1, 7]:
                r = train_target_only(target, k, seed)
                rows.append(r)
                print(f"[target-only] {DISPLAY[target]:<12} k={k:<4} seed={seed:<5} "
                      f"macro-F1 {r['test_macro_f1']:.4f}")
    print(f"\ntotal wall time {(time.time()-t_start)/60:.1f} min")

    target_only = pd.DataFrame(rows)
    target_only.to_csv(TABLES / "20_target_only_kshot_runs.csv", index=False)
    summ = (target_only.groupby(["target", "k"]).test_macro_f1
            .agg(["mean", "std", "count"]).reset_index())
    summ.to_csv(FT / "20_target_only_kshot_summary.csv", index=False)
    print("\nTarget-only control:")
    print(summ.to_string(index=False))

    le = AVAIL.get("label_eff_summary")
    if le is not None:
        src_init = pd.read_csv(le)
        f1c = _pick(src_init.columns, ["macro_f1_mean", "test_macro_f1"])
        mg = src_init.merge(summ.rename(columns={"mean": "target_only_mean"}),
                            on=["target", "k"], how="inner")
        mg["source_init_advantage"] = mg[f1c] - mg.target_only_mean
        mg = mg.rename(columns={f1c: "macro_f1_mean"})
        mg.to_csv(FT / "20_source_init_vs_target_only.csv", index=False)
        print("\nSource-initialised minus target-only, identical k:")
        print(mg[["source", "target", "k", "macro_f1_mean", "target_only_mean",
                  "source_init_advantage"]].to_string(index=False))
        adv = mg.source_init_advantage
        print(f"\nMean advantage of source initialisation: {adv.mean():+.4f}")
        print(f"Directions where it helps: {int((adv > 0).sum())}/{len(adv)}")
        if adv.mean() > 0.01:
            print("-> Source training contributes. 'Repair' is defensible in the title.")
        else:
            print("-> Source training contributes little. Keep 'Adaptation', and say so")
            print("   explicitly in Section 6.7 - it is a finding, not a weakness.")
elif RUN["T1_3_target_only"]:
    print("GPU not ready; skipped.")
else:
    print("skipped")

skipped


## T1-2 — XLM-R transfer matrix

Section 6.9 already shows the inversion reproducing under a TF-IDF linear classifier, and
the five-backbone comparison shows BanglaSarc highest in domain for every encoder. This
extends the *directed* matrix to a second transformer, which is what a referee asking about
encoder dependence actually wants.

15 trainings evaluated on three targets each, roughly 1.5–2 hours, about \$0.80.

In [26]:
xlmr = None
if RUN["T1_2_xlmr_transfer"] and GPU_READY:
    import inspect
    XLMR = "xlm-roberta-base"
    xt = AutoTokenizer.from_pretrained(XLMR, use_fast=True)

    class XDS(torch.utils.data.Dataset):
        def __init__(self, frame):
            self.enc = xt(list(frame.text.astype(str)), truncation=True,
                          max_length=MAX_LENGTH, padding="max_length")
            self.labels = frame.label_binary.values.astype(int)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, i):
            d = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
            d["labels"] = torch.tensor(self.labels[i])
            return d

    def mfn(p):
        return {"macro_f1": f1_score(p.label_ids, np.argmax(p.predictions, axis=1),
                                     average="macro", zero_division=0)}

    # ── Version-agnostic TrainingArguments ───────────────────────────────
    # 'evaluation_strategy' was renamed to 'eval_strategy' in transformers 4.41.
    # Build the kwargs, then keep only what this installed version accepts.
    _sig = set(inspect.signature(TrainingArguments.__init__).parameters)

    def make_args(out_dir, seed):
        want = dict(
            output_dir=str(out_dir),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            warmup_ratio=WARMUP_RATIO,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            save_total_limit=1,
            seed=seed,
            bf16=bool(HAS_BF16),
            fp16=bool(HAS_CUDA and not HAS_BF16),
            logging_steps=500,
            report_to=[],
            disable_tqdm=True,
        )
        # eval/save strategy under whichever name exists
        if "eval_strategy" in _sig:
            want["eval_strategy"] = "epoch"
        elif "evaluation_strategy" in _sig:
            want["evaluation_strategy"] = "epoch"
        if "save_strategy" in _sig:
            want["save_strategy"] = "epoch"
        # load_best_model_at_end requires matching eval and save strategies
        if not ({"eval_strategy", "evaluation_strategy"} & _sig):
            want["load_best_model_at_end"] = False
            want.pop("metric_for_best_model", None)
            want.pop("greater_is_better", None)
        return TrainingArguments(**{k: v for k, v in want.items() if k in _sig})

    _probe = make_args(CKPT / "_probe", 42)
    print("TrainingArguments built OK |",
          "eval_strategy" if "eval_strategy" in _sig else "evaluation_strategy",
          "| best-model selection:", getattr(_probe, "load_best_model_at_end", False))
    del _probe

    rows, t_start = [], time.time()
    for src in CORPORA:
        for seed in LOCO_SEEDS:
            set_seed(seed)
            model = AutoModelForSequenceClassification.from_pretrained(XLMR, num_labels=2)
            args = make_args(CKPT / f"xlmr_{src}_{seed}", seed)
            cbs = []
            if getattr(args, "load_best_model_at_end", False):
                cbs = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
            tkw = dict(model=model, args=args,
                       train_dataset=XDS(DATA[src]["train"]),
                       eval_dataset=XDS(DATA[src]["val"]),
                       compute_metrics=mfn, callbacks=cbs)
            # 'tokenizer' was renamed to 'processing_class' in transformers 4.46
            _tsig = set(inspect.signature(Trainer.__init__).parameters)
            if "processing_class" in _tsig:
                tkw["processing_class"] = xt
            elif "tokenizer" in _tsig:
                tkw["tokenizer"] = xt
            tr = Trainer(**tkw)
            t0 = time.time(); tr.train(); secs = float(time.time() - t0)
            for tgt in CORPORA:
                clean, removed = clean_target_frame([src], tgt)
                lg = np.asarray(tr.predict(XDS(clean)).predictions)
                if lg.ndim == 3:
                    lg = lg[0]
                m = score(clean.label_binary.values, lg.argmax(axis=1))
                rows.append(dict(encoder=XLMR, system="vanilla", source=src, target=tgt,
                                 seed=int(seed), in_domain=bool(src == tgt),
                                 n_removed_all_source=removed, n_eval_clean=len(clean),
                                 test_macro_f1=m["macro_f1"], test_accuracy=m["accuracy"],
                                 test_recall_class_1=m["recall_class_1"],
                                 train_seconds=secs))
                print(f"  XLM-R {DISPLAY[src]} -> {DISPLAY[tgt]} seed {seed}: "
                      f"{m['macro_f1']:.4f}")
            del tr, model
            free()
            try:
                import shutil
                shutil.rmtree(CKPT / f"xlmr_{src}_{seed}", ignore_errors=True)
            except Exception:
                pass
    print(f"\ntotal wall time {(time.time()-t_start)/60:.1f} min")

    xlmr = pd.DataFrame(rows)
    xlmr.to_csv(TABLES / "20_xlmr_transfer_runs.csv", index=False)
    summ = (xlmr.groupby(["source", "target", "in_domain"]).test_macro_f1
            .agg(["mean", "std", "count"]).reset_index())
    summ.to_csv(FT / "20_xlmr_transfer_summary.csv", index=False)
    ind = summ[summ.in_domain]["mean"].mean(); off = summ[~summ.in_domain]["mean"].mean()
    print(f"\nXLM-R in-domain {ind:.4f} | cross-corpus {off:.4f} | retention {off/ind:.4f}")
    print("BanglaBERT retention was 0.630 and TF-IDF 0.633.")
    print(summ.to_string(index=False))
elif RUN["T1_2_xlmr_transfer"]:
    print("GPU not ready; skipped.")
else:
    print("skipped")

TrainingArguments built OK | evaluation_strategy | best-model selection: True


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.6311, 'grad_norm': 4.8227128982543945, 'learning_rate': 1.949317738791423e-05, 'epoch': 0.7800312012480499}
{'eval_loss': 0.5308772325515747, 'eval_macro_f1': 0.7418706687426979, 'eval_runtime': 1.1435, 'eval_samples_per_second': 2240.54, 'eval_steps_per_second': 35.856, 'epoch': 1.0}
{'loss': 0.5448, 'grad_norm': 7.157726764678955, 'learning_rate': 1.7889490790899242e-05, 'epoch': 1.5600624024960998}
{'eval_loss': 0.5284696221351624, 'eval_macro_f1': 0.7527812827858248, 'eval_runtime': 1.1478, 'eval_samples_per_second': 2232.163, 'eval_steps_per_second': 35.722, 'epoch': 2.0}
{'loss': 0.4928, 'grad_norm': 7.240197658538818, 'learning_rate': 1.572264355362947e-05, 'epoch': 2.3400936037441498}
{'eval_loss': 0.5227373838424683, 'eval_macro_f1': 0.7712166752714469, 'eval_runtime': 1.1412, 'eval_samples_per_second': 2244.956, 'eval_steps_per_second': 35.926, 'epoch': 3.0}
{'loss': 0.4461, 'grad_norm': 11.735910415649414, 'learning_rate': 1.3555796316359698e-05, 'epoch': 3.120124

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.6312, 'grad_norm': 8.23859691619873, 'learning_rate': 1.949317738791423e-05, 'epoch': 0.7800312012480499}
{'eval_loss': 0.5724289417266846, 'eval_macro_f1': 0.7291101242287441, 'eval_runtime': 1.1515, 'eval_samples_per_second': 2224.846, 'eval_steps_per_second': 35.604, 'epoch': 1.0}
{'loss': 0.5531, 'grad_norm': 9.038244247436523, 'learning_rate': 1.7889490790899242e-05, 'epoch': 1.5600624024960998}
{'eval_loss': 0.4970563054084778, 'eval_macro_f1': 0.7563748079877113, 'eval_runtime': 1.1338, 'eval_samples_per_second': 2259.559, 'eval_steps_per_second': 36.16, 'epoch': 2.0}
{'loss': 0.4976, 'grad_norm': 8.711234092712402, 'learning_rate': 1.572264355362947e-05, 'epoch': 2.3400936037441498}
{'eval_loss': 0.5039777159690857, 'eval_macro_f1': 0.7719975057312712, 'eval_runtime': 1.1616, 'eval_samples_per_second': 2205.658, 'eval_steps_per_second': 35.297, 'epoch': 3.0}
{'loss': 0.4554, 'grad_norm': 17.780624389648438, 'learning_rate': 1.3555796316359698e-05, 'epoch': 3.12012480

...

{'eval_loss': 0.5948638916015625, 'eval_macro_f1': 0.7041336317135549, 'eval_runtime': 0.3583, 'eval_samples_per_second': 2207.456, 'eval_steps_per_second': 36.279, 'epoch': 1.0}
{'eval_loss': 0.562416672706604, 'eval_macro_f1': 0.7271017569534373, 'eval_runtime': 0.358, 'eval_samples_per_second': 2209.522, 'eval_steps_per_second': 36.313, 'epoch': 2.0}
{'loss': 0.6123, 'grad_norm': 6.997290134429932, 'learning_rate': 1.5214035087719299e-05, 'epoch': 2.525252525252525}
{'eval_loss': 0.5782992243766785, 'eval_macro_f1': 0.7383059418457648, 'eval_runtime': 0.3578, 'eval_samples_per_second': 2210.697, 'eval_steps_per_second': 36.333, 'epoch': 3.0}
{'eval_loss': 0.5541753768920898, 'eval_macro_f1': 0.7436476648849977, 'eval_runtime': 0.3579, 'eval_samples_per_second': 2209.897, 'eval_steps_per_second': 36.319, 'epoch': 4.0}
{'eval_loss': 0.6243365406990051, 'eval_macro_f1': 0.7356693448455051, 'eval_runtime': 0.3569, 'eval_samples_per_second': 2216.557, 'eval_steps_per_second': 36.429, 'ep

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'eval_loss': 0.6809992790222168, 'eval_macro_f1': 0.5997357627045464, 'eval_runtime': 0.3641, 'eval_samples_per_second': 2172.432, 'eval_steps_per_second': 35.704, 'epoch': 1.0}
{'eval_loss': 0.5744693875312805, 'eval_macro_f1': 0.7357707399327116, 'eval_runtime': 0.3567, 'eval_samples_per_second': 2217.314, 'eval_steps_per_second': 36.441, 'epoch': 2.0}
{'loss': 0.6096, 'grad_norm': 11.477158546447754, 'learning_rate': 1.5214035087719299e-05, 'epoch': 2.525252525252525}
{'eval_loss': 0.553676187992096, 'eval_macro_f1': 0.732178933568257, 'eval_runtime': 0.3563, 'eval_samples_per_second': 2220.193, 'eval_steps_per_second': 36.489, 'epoch': 3.0}
{'eval_loss': 0.5664424896240234, 'eval_macro_f1': 0.7420981971614883, 'eval_runtime': 0.3576, 'eval_samples_per_second': 2212.237, 'eval_steps_per_second': 36.358, 'epoch': 4.0}
{'eval_loss': 0.65560382604599, 'eval_macro_f1': 0.7431986578834157, 'eval_runtime': 0.3577, 'eval_samples_per_second': 2211.615, 'eval_steps_per_second': 36.348, 'epo

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'eval_loss': 0.6033526659011841, 'eval_macro_f1': 0.6797235060210839, 'eval_runtime': 0.3577, 'eval_samples_per_second': 2211.192, 'eval_steps_per_second': 36.341, 'epoch': 1.0}
{'eval_loss': 0.5686385631561279, 'eval_macro_f1': 0.7063962351693511, 'eval_runtime': 0.3572, 'eval_samples_per_second': 2214.491, 'eval_steps_per_second': 36.395, 'epoch': 2.0}
{'loss': 0.6077, 'grad_norm': 8.365976333618164, 'learning_rate': 1.5214035087719299e-05, 'epoch': 2.525252525252525}
{'eval_loss': 0.5346781611442566, 'eval_macro_f1': 0.7485026557585011, 'eval_runtime': 0.3617, 'eval_samples_per_second': 2187.162, 'eval_steps_per_second': 35.946, 'epoch': 3.0}
{'eval_loss': 0.5476369857788086, 'eval_macro_f1': 0.7383042688160184, 'eval_runtime': 0.3589, 'eval_samples_per_second': 2204.051, 'eval_steps_per_second': 36.223, 'epoch': 4.0}
{'eval_loss': 0.5937101244926453, 'eval_macro_f1': 0.754652605459057, 'eval_runtime': 0.3619, 'eval_samples_per_second': 2185.571, 'eval_steps_per_second': 35.92, 'ep

---

## Figures

Writes `F20_*.png` to `04_outputs/finalized_outputs/figures/`. Each block draws from an
in-memory result if the stage just ran, otherwise from the CSV on disk, otherwise it says
what is missing and moves on.

In [27]:
if RUN["FIGURES"]:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 11,
        "axes.titlesize": 13, "axes.labelsize": 11,
        "figure.dpi": 150, "savefig.dpi": 150,
        "axes.spines.top": False, "axes.spines.right": False})

    C_A, C_B, C_C, C_REF = "#1B7EC2", "#D95F02", "#7570B3", "#7F7F7F"
    made, skipped = [], []


    def pretty(x):
        k = str(x).replace("_binary", "")
        return {"ben_sarc": "Ben-Sarc", "banglasarc": "BanglaSarc",
                "banglasarc3": "BanglaSarc3"}.get(k, str(x))


    def grab(inmem, fname):
        """Prefer the in-memory frame, else read the CSV, else None."""
        if inmem is not None and len(inmem):
            return inmem
        for d in (FT, TABLES):
            p = d / fname
            if p.exists():
                try:
                    df = pd.read_csv(p)
                    if len(df):
                        return df
                except Exception:
                    pass
        return None


    def save(fig, name):
        p = FF / f"{name}.png"
        fig.savefig(p, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        made.append(name)
        print(f"  wrote {p.relative_to(ROOT)}")


    # ── F20-1  paired negative transfer ──────────────────────────────
    d = grab(paired if "paired" in dir() else None, "20_paired_negative_transfer.csv")
    if d is not None and "delta_pooled_minus_single" in d.columns:
        if "system" in d.columns and "fgm" in set(d.system.astype(str)):
            d = d[d.system.astype(str).eq("fgm")]
        d = d.sort_values("delta_pooled_minus_single").reset_index(drop=True)
        lbl = [f"+{pretty(r.added_source)} onto {pretty(r.base_source)}\n"
               f"(held-out {pretty(r.held_out_target)})" for _, r in d.iterrows()]
        y = np.arange(len(d))[::-1]
        err = np.vstack([d.delta_pooled_minus_single - d.delta_ci95_lo,
                         d.delta_ci95_hi - d.delta_pooled_minus_single])
        cols = [C_B if v < 0 else C_A for v in d.delta_pooled_minus_single]
        fig, ax = plt.subplots(figsize=(9.8, 0.92 * len(d) + 2.3))
        ax.errorbar(d.delta_pooled_minus_single, y, xerr=err, fmt="none",
                    capsize=4, lw=1.6, ecolor="black", zorder=3)
        ax.scatter(d.delta_pooled_minus_single, y, s=62, c=cols, zorder=4,
                   edgecolors="black", linewidths=0.8)
        ax.axvline(0, ls="--", color="black", lw=1.2)
        ax.set_yticks(y); ax.set_yticklabels(lbl, fontsize=9.2)
        ax.set_xlabel("Pooled minus single-source macro-F1 (paired over matched seeds)")
        ax.set_title("Adding a second source corpus: paired effect on held-out performance")
        ax.grid(axis="x", alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        save(fig, "F20_paired_negative_transfer")
    else:
        skipped.append("F20_paired_negative_transfer (needs T0-2)")

    # ── F20-2  collapse diagnostics ──────────────────────────────────
    d = grab(collapse if "collapse" in dir() else None, "20_collapse_diagnostics.csv")
    if d is not None and "auroc" in d.columns:
        sysname = "fgm" if "fgm" in set(d.system.astype(str)) else d.system.iloc[0]
        d = d[(~d.in_domain) & d.system.astype(str).eq(sysname)].sort_values(
            "macro_f1_argmax").reset_index(drop=True)
        lbl = [f"{pretty(r.source)}\n\u2192 {pretty(r.target)}" for _, r in d.iterrows()]
        x = np.arange(len(d)); w = 0.36
        fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.6),
                                 gridspec_kw={"width_ratios": [1.35, 1]})
        ax = axes[0]
        ax.bar(x - w / 2, d.macro_f1_argmax, w, label="Argmax (deployed)", color=C_A)
        ax.bar(x + w / 2, d.macro_f1_oracle_threshold, w, color=C_C,
               label="Target-oracle threshold (diagnostic ceiling)")
        ax.axhline(1 / 3, ls=":", color="firebrick", lw=1.3)
        ax.text(len(d) - 0.45, 1 / 3 + 0.015, "single-class floor \u2248 0.33",
                color="firebrick", fontsize=9, ha="right")
        ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=8.8)
        ax.set_ylabel("Macro-F1"); ax.set_ylim(0, 1.0)
        ax.set_title("Is the collapse a boundary-placement problem?")
        ax.legend(frameon=False, fontsize=9, loc="upper left")
        ax.grid(axis="y", alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        ax = axes[1]
        cols = ["firebrick" if v < 0.60 else C_A for v in d.auroc]
        ax.barh(np.arange(len(d))[::-1], d.auroc, 0.6, color=cols,
                edgecolor="black", lw=0.6)
        ax.axvline(0.5, ls="--", color="black", lw=1.2)
        ax.set_yticks(np.arange(len(d))[::-1]); ax.set_yticklabels(lbl, fontsize=8.8)
        ax.set_xlim(0, 1.0); ax.set_xlabel("AUROC (threshold-free)")
        ax.set_title("Does class ranking survive?")
        ax.grid(axis="x", alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        fig.suptitle("Class collapse: operating point versus ranking", fontsize=14)
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        save(fig, "F20_collapse_diagnostics")
    else:
        skipped.append("F20_collapse_diagnostics (needs item-level probabilities)")

    # ── F20-3  class-conditional length ──────────────────────────────
    d = grab(length_stats if "length_stats" in dir() else None,
             "20_class_conditional_length.csv")
    if d is not None and "char_cohens_d" in d.columns:
        tr = d[d.split.eq("train")].reset_index(drop=True)
        x = np.arange(len(tr)); w = 0.36
        fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.2))
        ax = axes[0]
        ax.bar(x - w / 2, tr.char_mean_nonsarc, w, label="Non-sarcastic", color=C_REF)
        ax.bar(x + w / 2, tr.char_mean_sarc, w, label="Sarcastic", color=C_B)
        ax.set_xticks(x); ax.set_xticklabels([pretty(c) for c in tr.corpus])
        ax.set_ylabel("Mean length (characters)")
        ax.set_title("Class-conditional length by corpus")
        ax.legend(frameon=False, fontsize=9.5)
        ax.grid(axis="y", alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        ax = axes[1]
        ax.bar(x, tr.char_cohens_d, 0.5,
               color=[C_B if abs(v) > 0.5 else C_A for v in tr.char_cohens_d],
               edgecolor="black", lw=0.6)
        ax.axhline(0, color="black", lw=1.0)
        for lvl, nm in ((0.2, "small"), (0.5, "medium"), (0.8, "large")):
            ax.axhline(lvl, ls=":", color="gray", lw=1.0)
            ax.text(len(tr) - 0.4, lvl + 0.012, nm, fontsize=8, color="gray", ha="right")
        ax.set_xticks(x); ax.set_xticklabels([pretty(c) for c in tr.corpus])
        ax.set_ylabel("Cohen's $d$ (sarcastic \u2212 non-sarcastic)")
        ax.set_title("Effect size of the length cue")
        ax.grid(axis="y", alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        sp = float(tr.char_cohens_d.max() - tr.char_cohens_d.min())
        fig.suptitle("Testing the length-shortcut mechanism across corpora", fontsize=14)
        fig.text(0.5, -0.04, f"Between-corpus spread in Cohen's $d$ is {sp:.3f}.",
                 ha="center", fontsize=9.3)
        fig.tight_layout(rect=[0, 0, 1, 0.93])
        save(fig, "F20_class_conditional_length")
    else:
        skipped.append("F20_class_conditional_length (needs T0-5)")

    # ── F20-4  common support ────────────────────────────────────────
    d = grab(common_support if "common_support" in dir() else None,
             "20_common_support_matrix.csv")
    if d is not None and "macro_f1_common_support" in d.columns:
        if "system" in d.columns and "fgm" in set(d.system.astype(str)):
            d = d[d.system.astype(str).eq("fgm")]
        d = d.reset_index(drop=True)
        lbl = [f"{pretty(r.source)}\n\u2192 {pretty(r.target)}" for _, r in d.iterrows()]
        x = np.arange(len(d)); w = 0.36
        fig, ax = plt.subplots(figsize=(11.0, 5.4))
        ax.bar(x - w / 2, d.macro_f1_own_filter, w, color=C_A,
               label="Own overlap filter (as published)")
        ax.bar(x + w / 2, d.macro_f1_common_support, w, color=C_C,
               label="Common support (identical items across sources)")
        for i, r in d.iterrows():
            ax.annotate(f"n {int(r.n_own_filter)}\u2192{int(r.n_common_support)}",
                        (i, max(r.macro_f1_own_filter, r.macro_f1_common_support) + 0.02),
                        ha="center", fontsize=8, color="dimgray")
        ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=8.8)
        ax.set_ylabel("Macro-F1"); ax.set_ylim(0, 1.02)
        ax.set_title("Transfer results are unchanged on a common-support evaluation set")
        ax.legend(frameon=False, fontsize=9.5, loc="upper left")
        ax.grid(axis="y", alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        fig.text(0.5, -0.03,
                 f"Largest absolute change: {d.delta.abs().max():.4f} macro-F1.",
                 ha="center", fontsize=9.3)
        fig.tight_layout()
        save(fig, "F20_common_support")
    else:
        skipped.append("F20_common_support (needs item-level predictions)")

    # ── F20-5  budget-matched DANN ───────────────────────────────────
    d = grab(None, "20_dann_matched_vs_pooled.csv")
    if d is not None and "delta_vs_pooled" in d.columns:
        lbl = [pretty(c) for c in d.held_out_corpus]
        x = np.arange(len(d)); w = 0.26
        fig, ax = plt.subplots(figsize=(9.8, 5.8))
        ax.bar(x - w, d["macro_f1_mean"], w, label="Pooled LOCO (8 epochs)", color=C_A)
        ax.bar(x, d["mean"], w, yerr=d.get("std"), capsize=3, ecolor="black",
               label="DANN, budget-matched (8 epochs)", color=C_C)
        p_old = AVAIL.get("dann")
        if p_old is not None:
            old = pd.read_csv(p_old)
            if "held_out" in old.columns:
                old = old[old.held_out.astype(bool)]
            g = old.groupby("held_out_corpus").test_macro_f1.mean()
            vals = [g.get(c, np.nan) for c in d.held_out_corpus]
            ax.bar(x + w, vals, w, label="DANN as published (4 epochs)",
                   color=C_REF, hatch="//", edgecolor="white")
        ax.set_xticks(x); ax.set_xticklabels(lbl)
        ax.set_xlabel("Held-out target corpus"); ax.set_ylabel("Macro-F1")
        ax.set_title("Domain-adversarial training at a matched optimisation budget")
        ax.legend(frameon=False, fontsize=9.5)
        ax.grid(axis="y", alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        fig.text(0.5, -0.03,
                 f"Mean difference from pooled at matched budget: "
                 f"{d.delta_vs_pooled.mean():+.4f} macro-F1 "
                 f"(published value was \u22120.0216 at half the budget).",
                 ha="center", fontsize=9.3)
        fig.tight_layout()
        save(fig, "F20_dann_budget_matched")
    else:
        skipped.append("F20_dann_budget_matched (needs T1-1)")

    # ── F20-6  source-init vs target-only ────────────────────────────
    d = grab(None, "20_source_init_vs_target_only.csv")
    if d is not None and "target_only_mean" in d.columns:
        tg = [t for t in CORPORA if t in set(d.target)]
        fig, axes = plt.subplots(1, max(1, len(tg)), figsize=(4.7 * max(1, len(tg)), 5.2),
                                 sharey=True, squeeze=False)
        for ax, t in zip(axes[0], tg):
            sub = d[d.target.eq(t)]
            ks = sorted(sub.k.unique()); pos = np.arange(len(ks))
            for i, (s, g) in enumerate(sorted(sub.groupby("source"))):
                g = g.sort_values("k")
                ax.plot(pos[:len(g)], g.macro_f1_mean, marker="o", ms=5, lw=1.7,
                        color=[C_A, C_B][i % 2], label=f"from {pretty(s)}")
            to = sub.groupby("k").target_only_mean.mean().reindex(ks)
            ax.plot(pos, to.values, marker="s", ms=5, lw=1.7, ls="--", color="black",
                    label="target-only (no source)")
            ax.set_xticks(pos); ax.set_xticklabels([str(k) for k in ks])
            ax.set_xlabel("Labelled target examples $k$")
            ax.set_title(f"Target: {pretty(t)}", fontsize=12)
            ax.legend(frameon=False, fontsize=8.6, loc="lower right")
            ax.grid(alpha=0.22, lw=0.6); ax.set_axisbelow(True)
        axes[0][0].set_ylabel("Macro-F1")
        fig.suptitle("Does source training contribute, or do the target labels do the work?",
                     fontsize=13.5)
        fig.text(0.5, -0.03,
                 f"Mean advantage of source initialisation: "
                 f"{d.source_init_advantage.mean():+.4f} macro-F1.",
                 ha="center", fontsize=9.3)
        fig.tight_layout(rect=[0, 0, 1, 0.93])
        save(fig, "F20_source_init_vs_target_only")
    else:
        skipped.append("F20_source_init_vs_target_only (needs T1-3)")

    # ── F20-7  model families ────────────────────────────────────────
    fams = []
    p = AVAIL.get("single_summary")
    if p is not None:
        b = pd.read_csv(p)
        if "system" in b.columns and "fgm" in set(b.system.astype(str)):
            b = b[b.system.astype(str).eq("fgm")]
        f1c = _pick(b.columns, ["macro_f1_mean", "test_macro_f1"])
        if f1c and "in_domain" in b.columns:
            fams.append(("BanglaBERT", b[b.in_domain][f1c].mean(),
                         b[~b.in_domain][f1c].mean()))
    px = FT / "20_xlmr_transfer_summary.csv"
    if px.exists():
        x_ = pd.read_csv(px)
        fams.append(("XLM-RoBERTa", x_[x_.in_domain]["mean"].mean(),
                     x_[~x_.in_domain]["mean"].mean()))
    p = AVAIL.get("classical")
    if p is not None:
        c = pd.read_csv(p)
        f1c = _pick(c.columns, ["macro_f1", "test_macro_f1"])
        if f1c and "in_domain" in c.columns:
            fams.append(("TF-IDF + linear", c[c.in_domain][f1c].mean(),
                         c[~c.in_domain][f1c].mean()))
    if len(fams) >= 2:
        names = [f[0] for f in fams]
        ind = np.array([f[1] for f in fams]); off = np.array([f[2] for f in fams])
        x = np.arange(len(fams)); w = 0.34
        fig, axes = plt.subplots(1, 2, figsize=(12.6, 5.4),
                                 gridspec_kw={"width_ratios": [1.25, 1]})
        ax = axes[0]
        ax.bar(x - w / 2, ind, w, label="In-domain (mean of 3 cells)", color=C_A)
        ax.bar(x + w / 2, off, w, label="Cross-corpus (mean of 6 cells)", color=C_B)
        ax.set_xticks(x); ax.set_xticklabels(names)
        ax.set_ylabel("Macro-F1"); ax.set_ylim(0, 1.02)
        ax.set_title("The gap is present in every model family")
        ax.legend(frameon=False, fontsize=9.5)
        ax.grid(axis="y", alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        ax = axes[1]
        ret = off / ind
        ax.bar(x, ret, 0.5, color=C_C, edgecolor="black", lw=0.6)
        for i, v in enumerate(ret):
            ax.annotate(f"{v:.3f}", (i, v + 0.012), ha="center", fontsize=10)
        ax.set_xticks(x); ax.set_xticklabels(names)
        ax.set_ylim(0, max(0.85, float(ret.max()) + 0.12))
        ax.set_ylabel("Retention (cross-corpus / in-domain)")
        ax.set_title("Retention is nearly identical across families")
        ax.grid(axis="y", alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        fig.tight_layout()
        save(fig, "F20_model_families")
    else:
        skipped.append("F20_model_families (needs at least two model families)")

    print(f"\n{len(made)} figure(s) in {FF.relative_to(ROOT)}")
    for m in made:
        print("   +", m)
    if skipped:
        print("\nNot produced:")
        for s in skipped:
            print("   -", s)
else:
    print("figures disabled")

  wrote 04_outputs/finalized_outputs/figures/F20_paired_negative_transfer.png
  wrote 04_outputs/finalized_outputs/figures/F20_collapse_diagnostics.png
  wrote 04_outputs/finalized_outputs/figures/F20_class_conditional_length.png
  wrote 04_outputs/finalized_outputs/figures/F20_common_support.png
  wrote 04_outputs/finalized_outputs/figures/F20_dann_budget_matched.png
  wrote 04_outputs/finalized_outputs/figures/F20_source_init_vs_target_only.png
  wrote 04_outputs/finalized_outputs/figures/F20_model_families.png

7 figure(s) in 04_outputs/finalized_outputs/figures
   + F20_paired_negative_transfer
   + F20_collapse_diagnostics
   + F20_class_conditional_length
   + F20_common_support
   + F20_dann_budget_matched
   + F20_source_init_vs_target_only
   + F20_model_families


## Manifest

Checksums every table and figure this notebook produced, and maps each to the manuscript
section it affects.

In [28]:
written = sorted(set(list(FT.glob("20_*.csv")) + list(TABLES.glob("20_*.csv")) +
                     list(FT.glob("20_*.txt")) + list(FF.glob("F20_*.png"))))
rows = []
for f in written:
    try:
        n = len(pd.read_csv(f)) if f.suffix == ".csv" else None
    except Exception:
        n = None
    rows.append(dict(path=str(f.relative_to(ROOT)), rows=n, bytes=f.stat().st_size,
                     sha256=hashlib.sha256(f.read_bytes()).hexdigest()))
manifest = pd.DataFrame(rows)
if len(manifest):
    manifest.to_csv(FT / "20_MANIFEST_sha256.csv", index=False)
    print(manifest[["path", "rows", "bytes"]].to_string(index=False))
else:
    print("Nothing produced yet.")

MAPPING = {
 "20_common_support_matrix.csv":       "6.2  - replaces the nested-population wording",
 "20_common_support_populations.csv":  "6.2  - population counts when predictions are absent",
 "20_paired_negative_transfer.csv":    "6.5, C3, abstract - makes 15.8 points paired",
 "20_collapse_diagnostics.csv":        "6.3, 6.6 - boundary versus ranking",
 "20_class_conditional_length.csv":    "6.3  - removes 'we did not compute those'",
 "20_regenerated_summary.csv":         "5, C5 - tables regenerate from predictions",
 "20_data_availability_wording.txt":   "5, 8, Data availability - CORRECTED TEXT, paste it",
 "20_dann_matched_vs_pooled.csv":      "4.4, 6.6, 9 - removes the epoch-budget confound",
 "20_source_init_vs_target_only.csv":  "6.7, 8, title - licenses 'Repair' or confirms 'Adaptation'",
 "20_xlmr_transfer_summary.csv":       "6.9  - second transformer encoder",
}
print("\nManuscript impact:")
for k, v in MAPPING.items():
    present = (FT / k).exists() or (TABLES / k).exists()
    print(f"  [{'done   ' if present else 'not run'}] {k:<38} {v}")

print("\n" + "=" * 74)
print("NEXT STEP: open 20_data_availability_wording.txt and paste its two paragraphs")
print("into Section 5 and the Data-availability statement. The current manuscript text")
print("promises item-level predictions that are not in the repository.")
print("=" * 74)

                                                                   path  rows  bytes
  04_outputs/finalized_outputs/figures/F20_class_conditional_length.png   NaN  87281
      04_outputs/finalized_outputs/figures/F20_collapse_diagnostics.png   NaN 117281
            04_outputs/finalized_outputs/figures/F20_common_support.png   NaN  72740
       04_outputs/finalized_outputs/figures/F20_dann_budget_matched.png   NaN  84957
            04_outputs/finalized_outputs/figures/F20_model_families.png   NaN  70371
  04_outputs/finalized_outputs/figures/F20_paired_negative_transfer.png   NaN 101156
04_outputs/finalized_outputs/figures/F20_source_init_vs_target_only.png   NaN 120220
             04_outputs/finalized_outputs/tables/20_MANIFEST_sha256.csv  12.0   1686
    04_outputs/finalized_outputs/tables/20_class_conditional_length.csv   6.0   1110
        04_outputs/finalized_outputs/tables/20_collapse_diagnostics.csv  27.0   5116
       04_outputs/finalized_outputs/tables/20_common_support_matr